# MCQ Generator Using Google Gemini API and LangChain

This notebook demonstrates how to create a Multiple Choice Question (MCQ) generator using Google Gemini API with LangChain framework.

## Features:
- Generate MCQs from any text input
- Customizable number of questions, subject, and tone
- Question evaluation and improvement
- Export results to CSV format

In [74]:
# Install required packages
%pip install langchain-google-genai google-generativeai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [75]:
# Import required libraries
import os
import pandas as pd
import json
import traceback
from datetime import datetime

In [76]:
# Import LangChain and Google Gemini components
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain.schema import HumanMessage
from dotenv import load_dotenv

In [ ]:
# Load environment variables and setup Google Gemini API
load_dotenv()

# Get Google API key from environment or set it directly
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# If not in environment, set it directly (replace with your actual key)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = "you_own_api_key_here"  # Replace with your actual Google API key
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Initialize the Gemini model
model_name = "gemini-1.5-pro-latest"  # Update this if your model is different

llm = ChatGoogleGenerativeAI(
    model=model_name, 
    temperature=0.7,
    google_api_key=GOOGLE_API_KEY
)

print(f"✅ Google Gemini API initialized with model: {model_name}")

✅ Google Gemini API initialized with model: gemini-1.5-pro-latest


In [78]:
# Test the connection with a simple query using the updated model
try:
    test_response = llm.invoke([
        HumanMessage(content="What is machine learning in simple terms?")
    ])
    print("🧪 Test Response:")
    print(test_response.content)
except Exception as e:
    print(f"❌ Error: {e}")

🧪 Test Response:
Imagine you have a puppy you're trying to teach to sit.  You show them what "sit" means, give them treats when they do it right, and correct them when they don't. Over time, the puppy learns to connect the word "sit" with the action, and eventually sits on command reliably.

Machine learning is similar.  Instead of a puppy, we have a computer program. Instead of treats and corrections, we have data. We feed the program tons of data and tell it what to look for (like showing the puppy what "sit" looks like). The program then finds patterns in the data and builds its own set of rules (like the puppy learning the connection between "sit" and the action).  This allows the program to make predictions or decisions on new data it hasn't seen before (like the puppy sitting on command even in a new location).

So basically, machine learning is teaching computers to learn from data without explicitly programming them with every single rule.


In [99]:
# Define the response format template
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here", 
            "c": "choice here",
            "d": "choice here"
        },
        "correct": "correct answer",
        "explanation": "explanation of the correct answer"
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here", 
            "d": "choice here"
        },
        "correct": "correct answer",
        "explanation": "explanation of the correct answer"
    }
}

print("📝 Response format template defined")

📝 Response format template defined


In [100]:
# Define the MCQ generation template
TEMPLATE = """
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.

Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide.
Ensure to make {number} MCQs only.

### RESPONSE_JSON
{response_json}

### INSTRUCTIONS:
1. Create exactly {number} questions
2. Each question should have 4 options (a, b, c, d)
3. Clearly indicate the correct answer
4. Provide explanation for each correct answer
5. Questions should be relevant to the given text
6. Use {tone} tone throughout
7. Target {subject} students level

"""

print("📋 MCQ generation template created")

📋 MCQ generation template created


In [101]:
# Create the quiz generator prompt
quiz_generator_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

print("🔧 Quiz generator prompt configured")

🔧 Quiz generator prompt configured


In [102]:
# Create the quiz generation chain
quiz_chain = LLMChain(
    llm=llm,
    prompt=quiz_generator_prompt,
    output_key="quiz",
    verbose=True
)

print("⚡ Quiz generation chain created")

⚡ Quiz generation chain created


In [103]:
# Define the evaluation template
TEMPLATE2 = """
You are an expert English grammarian and writer. Given a Multiple Choice Quiz for {subject} students.
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis.

If the quiz is not at par with the cognitive and analytical abilities of the students, update the quiz questions which need to be changed and change the tone such that it perfectly fits the student abilities.

Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

print("📊 Evaluation template defined")

📊 Evaluation template defined


In [104]:
# Create the quiz evaluation prompt
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["quiz", "subject"],
    template=TEMPLATE2
)

print("🔍 Quiz evaluation prompt configured")

🔍 Quiz evaluation prompt configured


In [105]:
# Create the review chain
review_chain = LLMChain(
    llm=llm,
    prompt=quiz_evaluation_prompt,
    output_key="review",
    verbose=True
)

print("📝 Review chain created")

📝 Review chain created


In [106]:
# Create the sequential chain that combines quiz generation and evaluation
generate_evaluation_chain = SequentialChain(
    chains=[quiz_chain, review_chain],
    input_variables=["text", "number", "subject", "tone", "response_json"],
    output_variables=["quiz", "review"],
    verbose=True
)

print("🔗 Sequential chain configured")

🔗 Sequential chain configured


## Sample Text for MCQ Generation

Let's use a sample text about machine learning to generate MCQs.

In [87]:
# Sample text about machine learning
SAMPLE_TEXT = """
The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence. The synonym self-teaching computers was also used in this time period.

The earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes. In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells. Hebb's model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data.

By the early 1960s, an experimental "learning machine" with punched tape memory, called Cybertron, had been developed by Raytheon Company to analyse sonar signals, electrocardiograms, and speech patterns using rudimentary reinforcement learning. It was repetitively "trained" by a human operator/teacher to recognise patterns and equipped with a "goof" button to cause it to reevaluate incorrect decisions.

Modern-day machine learning has two objectives. One is to classify data based on models which have been developed; the other purpose is to make predictions for future outcomes based on these models. A hypothetical algorithm specific to classifying data may use computer vision of moles coupled with supervised learning in order to train it to classify the cancerous moles. A machine learning algorithm for stock trading may inform the trader of future potential predictions.
"""

print("📖 Sample text loaded")
print(f"Text length: {len(SAMPLE_TEXT)} characters")

📖 Sample text loaded
Text length: 1794 characters


In [88]:
# Set parameters for MCQ generation
NUMBER = 5
SUBJECT = "machine learning"
TONE = "simple"

print(f"🎯 Parameters set:")
print(f"Number of questions: {NUMBER}")
print(f"Subject: {SUBJECT}")
print(f"Tone: {TONE}")

🎯 Parameters set:
Number of questions: 5
Subject: machine learning
Tone: simple


In [89]:
# Generate MCQs using the complete chain
print("🚀 Starting MCQ generation...")

try:
    response = generate_evaluation_chain(
        {
            "text": SAMPLE_TEXT,
            "number": NUMBER,
            "subject": SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
    )
    
    print("\n✅ MCQ Generation completed successfully!")
    print(f"Generated quiz length: {len(response.get('quiz', ''))} characters")
    print(f"Generated review length: {len(response.get('review', ''))} characters")
    
except Exception as e:
    print(f"❌ Error during generation: {str(e)}")
    print("Full traceback:")
    traceback.print_exc()

🚀 Starting MCQ generation...


> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text: 
The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence. The synonym self-teaching computers was also used in this time period.

The earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes. In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells. Hebb's model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial

In [90]:
# Display the generated response
print("📋 Generated MCQs and Review:")
print("="*50)
print(response)

📋 Generated MCQs and Review:
{'text': '\nThe term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence. The synonym self-teaching computers was also used in this time period.\n\nThe earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes. In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells. Hebb\'s model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data.\n\nBy the early 1960s, an experimental "learnin

In [91]:
# Extract and parse the quiz
try:
    quiz_content = response.get("quiz")
    print("🧩 Raw Quiz Content:")
    print(quiz_content)
    
    # Try to extract JSON from the response
    import re
    json_match = re.search(r'\{.*\}', quiz_content, re.DOTALL)
    if json_match:
        quiz_json = json.loads(json_match.group())
        print("\n✅ Successfully parsed quiz JSON")
    else:
        print("⚠️ Could not find JSON in response, using raw content")
        quiz_json = {}
        
except Exception as e:
    print(f"❌ Error parsing quiz: {str(e)}")
    quiz_json = {}

🧩 Raw Quiz Content:
```json
{
  "1": {
    "mcq": "Who coined the term 'machine learning'?",
    "options": {
      "a": "Donald Hebb",
      "b": "Raytheon Company",
      "c": "Arthur Samuel",
      "d": "Alan Turing"
    },
    "correct": "c",
    "explanation": "Arthur Samuel, an IBM employee, coined the term 'machine learning' in 1959."
  },
  "2": {
    "mcq": "What was the purpose of Arthur Samuel's early machine learning program?",
    "options": {
      "a": "Analyzing sonar signals",
      "b": "Classifying cancerous moles",
      "c": "Calculating winning chances in checkers",
      "d": "Predicting stock prices"
    },
    "correct": "c",
    "explanation": "Arthur Samuel's program, one of the earliest examples of machine learning, was designed to calculate the likelihood of winning in a game of checkers."
  },
  "3": {
    "mcq": "What was the name of the experimental 'learning machine' developed by Raytheon Company?",
    "options": {
      "a": "HebbNet",
      "b": "Cyb

In [92]:
# Convert quiz to structured format for display
if quiz_json:
    quiz_table_data = []
    
    for key, value in quiz_json.items():
        mcq = value.get("mcq", "")
        options = value.get("options", {})
        correct = value.get("correct", "")
        explanation = value.get("explanation", "")
        
        # Format options as a readable string
        options_str = " | ".join([
            f"{option}: {option_value}"
            for option, option_value in options.items()
        ])
        
        quiz_table_data.append({
            "Question": mcq,
            "Options": options_str,
            "Correct Answer": correct,
            "Explanation": explanation
        })
    
    print(f"📊 Structured {len(quiz_table_data)} questions")
else:
    quiz_table_data = []
    print("⚠️ No structured quiz data available")

📊 Structured 5 questions


In [93]:
# Display the quiz in a readable format
if quiz_table_data:
    print("📝 Generated MCQs:")
    print("="*80)
    
    for i, question in enumerate(quiz_table_data, 1):
        print(f"\nQuestion {i}: {question['Question']}")
        print(f"Options: {question['Options']}")
        print(f"Correct Answer: {question['Correct Answer']}")
        print(f"Explanation: {question['Explanation']}")
        print("-" * 60)
else:
    print("⚠️ No questions to display")

📝 Generated MCQs:

Question 1: Who coined the term 'machine learning'?
Options: a: Donald Hebb | b: Raytheon Company | c: Arthur Samuel | d: Alan Turing
Correct Answer: c
Explanation: Arthur Samuel, an IBM employee, coined the term 'machine learning' in 1959.
------------------------------------------------------------

Question 2: What was the purpose of Arthur Samuel's early machine learning program?
Options: a: Analyzing sonar signals | b: Classifying cancerous moles | c: Calculating winning chances in checkers | d: Predicting stock prices
Correct Answer: c
Explanation: Arthur Samuel's program, one of the earliest examples of machine learning, was designed to calculate the likelihood of winning in a game of checkers.
------------------------------------------------------------

Question 3: What was the name of the experimental 'learning machine' developed by Raytheon Company?
Options: a: HebbNet | b: Cybertron | c: SamuelCheckers | d: DeepThought
Correct Answer: b
Explanation: Rayth

In [94]:
# Save results to CSV file
if quiz_table_data:
    # Create DataFrame
    quiz_df = pd.DataFrame(quiz_table_data)
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"mcq_{SUBJECT.replace(' ', '_')}_{timestamp}.csv"
    
    # Save to CSV
    quiz_df.to_csv(filename, index=False)
    
    print(f"💾 Quiz saved to: {filename}")
    print(f"📊 Saved {len(quiz_table_data)} questions")
    
    # Display the DataFrame
    print("\n📋 Quiz DataFrame:")
    print(quiz_df)
else:
    print("⚠️ No data to save")

💾 Quiz saved to: mcq_machine_learning_20250619_083425.csv
📊 Saved 5 questions

📋 Quiz DataFrame:
                                            Question  \
0            Who coined the term 'machine learning'?   
1  What was the purpose of Arthur Samuel's early ...   
2  What was the name of the experimental 'learnin...   
3  What are the two main objectives of modern mac...   
4  Donald Hebb's work contributed to understandin...   

                                             Options Correct Answer  \
0  a: Donald Hebb | b: Raytheon Company | c: Arth...              c   
1  a: Analyzing sonar signals | b: Classifying ca...              c   
2  a: HebbNet | b: Cybertron | c: SamuelCheckers ...              b   
3  a: Playing games and analyzing images | b: Cla...              b   
4  a: Computer vision | b: Stock trading algorith...              c   

                                         Explanation  
0  Arthur Samuel, an IBM employee, coined the ter...  
1  Arthur Samuel's program, o

In [95]:
# Display the review/evaluation
if response.get("review"):
    print("📊 Quiz Review and Evaluation:")
    print("="*50)
    print(response.get("review"))
else:
    print("⚠️ No review available")

📊 Quiz Review and Evaluation:
Complexity Analysis:

The quiz exhibits moderate complexity.  It assumes familiarity with key figures and foundational concepts in machine learning history. Some options require nuanced understanding.


Revised Quiz:

```json
{
  "1": {
    "mcq": "The term 'machine learning' was coined by which pioneering figure?",
    "options": {
      "a": "Donald Hebb, known for his work on neural networks",
      "b": "Raytheon Company, a defense technology company",
      "c": "Arthur Samuel, a computer scientist at IBM",
      "d": "Alan Turing, considered the father of theoretical computer science and AI"
    },
    "correct": "c",
    "explanation": "Arthur Samuel, while working at IBM in 1959, introduced the term 'machine learning' to describe the ability of computers to learn without explicit programming."
  },
  "2": {
    "mcq": "Arthur Samuel's groundbreaking machine learning program focused on which application?",
    "options": {
      "a": "Analyzing comp

## Custom MCQ Generation

You can now generate MCQs from your own text by modifying the parameters below.

In [96]:
# Function to generate MCQs with custom parameters
def generate_custom_mcqs(text, num_questions=5, subject="general", tone="simple"):
    """
    Generate MCQs with custom parameters
    
    Args:
        text (str): The text to generate questions from
        num_questions (int): Number of questions to generate
        subject (str): Subject area for the questions
        tone (str): Tone of the questions (simple, moderate, complex)
    
    Returns:
        dict: Generated response with quiz and review
    """
    try:
        print(f"🎯 Generating {num_questions} {subject} questions in {tone} tone...")
        
        response = generate_evaluation_chain(
            {
                "text": text,
                "number": num_questions,
                "subject": subject,
                "tone": tone,
                "response_json": json.dumps(RESPONSE_JSON)
            }
        )
        
        print("✅ Custom MCQ generation completed!")
        return response
        
    except Exception as e:
        print(f"❌ Error generating custom MCQs: {str(e)}")
        return None

# Example usage:
# custom_text = "Your text here..."
# custom_response = generate_custom_mcqs(custom_text, 3, "science", "moderate")

In [97]:
# Load text from file (optional)
def load_text_from_file(file_path):
    """
    Load text from a file for MCQ generation
    
    Args:
        file_path (str): Path to the text file
    
    Returns:
        str: Content of the file
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        print(f"📖 Loaded text from {file_path}")
        print(f"Text length: {len(content)} characters")
        return content
    except Exception as e:
        print(f"❌ Error loading file: {str(e)}")
        return None

# Example usage:
# text_content = load_text_from_file("/path/to/your/file.txt")
# if text_content:
#     response = generate_custom_mcqs(text_content, 10, "history", "moderate")

## Summary

This MCQ generator provides:

1. **Google Gemini Integration**: Uses Google's Gemini Pro model for high-quality question generation
2. **LangChain Framework**: Leverages LangChain for structured prompt engineering and chain management
3. **Customizable Parameters**: Adjustable number of questions, subject area, and tone
4. **Quality Evaluation**: Built-in review system to assess and improve question quality
5. **Export Functionality**: Save results to CSV format for easy sharing and analysis
6. **Error Handling**: Robust error handling for reliable operation

### Usage Tips:
- Replace `"your-google-api-key-here"` with your actual Google API key
- Adjust the `NUMBER`, `SUBJECT`, and `TONE` variables as needed
- Use the custom functions to generate MCQs from your own text files
- The system works best with well-structured, informative text content

### Next Steps:
- Add more question types (True/False, Fill-in-the-blank)
- Implement difficulty levels
- Add support for images in questions
- Create a web interface using Streamlit

In [98]:
# List available Gemini models and their supported methods
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)

models = genai.list_models()
print('Available models:')
for m in models:
    print(f"- {m.name}: {m.supported_generation_methods}")

Available models:
- models/embedding-gecko-001: ['embedText', 'countTextTokens']
- models/gemini-1.0-pro-vision-latest: ['generateContent', 'countTokens']
- models/gemini-pro-vision: ['generateContent', 'countTokens']
- models/gemini-1.5-pro-latest: ['generateContent', 'countTokens']
- models/gemini-1.5-pro-002: ['generateContent', 'countTokens', 'createCachedContent']
- models/gemini-1.5-pro: ['generateContent', 'countTokens']
- models/gemini-1.5-flash-latest: ['generateContent', 'countTokens']
- models/gemini-1.5-flash: ['generateContent', 'countTokens']
- models/gemini-1.5-flash-002: ['generateContent', 'countTokens', 'createCachedContent']
- models/gemini-1.5-flash-8b: ['createCachedContent', 'generateContent', 'countTokens']
- models/gemini-1.5-flash-8b-001: ['createCachedContent', 'generateContent', 'countTokens']
- models/gemini-1.5-flash-8b-latest: ['createCachedContent', 'generateContent', 'countTokens']
- models/gemini-2.5-pro-exp-03-25: ['generateContent', 'countTokens', 'cr